In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import heapq
import time
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from collections import deque
from tqdm import tqdm
import warnings

torch.set_num_threads(1)
warnings.filterwarnings('ignore')

# 1. КОНФИГУРАЦИЯ ЭКСПЕРИМЕНТА
GRID_SIZE = 30
PATCH_SIZE = 30
DENSITIES = np.arange(0.10, 0.55, 0.10) 
TOTAL_MAPS_PER_DENSITY = 2500 # Чтобы получить 500 тестовых карт при 20% сплите
TEST_FRACTION = 0.2
SEED = 42
DEVICE = torch.device('cpu') 

# 2. ГЕНЕРАЦИЯ СРЕДЫ И ДАННЫХ
def generate_map(size, density, rng):
    noise = rng.random((size, size)) < (density * 0.5)
    seeds = rng.random((size, size)) < (density * 0.1)
    clusters = binary_dilation(seeds, iterations=3)
    
    grid = noise | clusters
    grid[0, 0] = 0
    grid[size-1, size-1] = 0
    return grid.astype(np.uint8)

def is_reachable_8way(grid, start, goal):
    size = grid.shape[0]
    visited = np.zeros((size, size), dtype=bool)
    queue = deque([start])
    visited[start] = True
    
    while queue:
        x, y = queue.popleft()
        if (x, y) == goal: return True
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0: continue
                nx, ny = x + dx, y + dy
                if 0 <= nx < size and 0 <= ny < size and not visited[nx, ny] and grid[nx, ny] == 0:
                    if dx != 0 and dy != 0:
                        if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                    visited[nx, ny] = True
                    queue.append((nx, ny))
    return False

def compute_dijkstra_heuristic(grid, goal):
    size = grid.shape[0]
    h_star = np.full((size, size), np.inf)
    h_star[goal] = 0
    pq = [(0, goal)]
    
    while pq:
        cost, (x, y) = heapq.heappop(pq)
        if cost > h_star[x, y]: continue
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0: continue
                nx, ny = x + dx, y + dy
                if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                    if dx != 0 and dy != 0:
                        if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                    move_cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                    new_cost = cost + move_cost
                    if new_cost < h_star[nx, ny]:
                        h_star[nx, ny] = new_cost
                        heapq.heappush(pq, (new_cost, (nx, ny)))
    return h_star


# 3. CNN МОДЕЛЬ И DATASET
class HeuristicDataset(Dataset):
    def __init__(self, maps, h_stars, patch_size):
        self.samples = []
        self.patch_size = patch_size
        pad = patch_size // 2
        
        for grid, h_star in zip(maps, h_stars):
            size = grid.shape[0]
            goal = (size-1, size-1)
            y_coords, x_coords = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
            dist_to_goal = np.sqrt((x_coords - goal[1])**2 + (y_coords - goal[0])**2)
            dist_to_goal = dist_to_goal / np.max(dist_to_goal)
            
            padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
            padded_dist = np.pad(dist_to_goal, pad, mode='constant', constant_values=1.0)
            
            for x in range(size):
                for y in range(size):
                    if grid[x, y] == 0 and h_star[x, y] != np.inf:
                        patch_obs = padded_grid[x:x+patch_size, y:y+patch_size]
                        patch_dist = padded_dist[x:x+patch_size, y:y+patch_size]
                        sample = np.stack([patch_obs, patch_dist], axis=0)
                        self.samples.append((sample, h_star[x, y]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

class LightweightCNN(nn.Module):
    def __init__(self):
        super(LightweightCNN, self).__init__()
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        return self.fc(torch.flatten(x, 1))

# Проверка числа параметров 
model_check = LightweightCNN()
print(f" Реальное число параметров CNN: {sum(p.numel() for p in model_check.parameters()):,}")

# 4. АЛГОРИТМЫ ПОИСКА ПУТИ
def get_neighbors_8way(grid, current):
    x, y = current
    size = grid.shape[0]
    neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                if dx != 0 and dy != 0:
                    if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                neighbors.append(((nx, ny), cost))
    return neighbors

def euclidean_heuristic(node, goal):
    return np.sqrt((node[0] - goal[0])**2 + (node[1] - goal[1])**2)

def astar_search(grid, start, goal, epsilon=1.0, cnn_model=None, patch_size=30):
    size = grid.shape[0]
    open_set = []
    heapq.heappush(open_set, (0, 0, 0, start)) 
    g_score = {start: 0}
    nodes_expanded = 0
    total_inf_time = 0.0
    counter = 1
    
    if cnn_model is not None:
        pad = patch_size // 2
        padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
        y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        dist_map = np.sqrt((x_c - goal[1])**2 + (y_c - goal[0])**2)
        dist_map = dist_map / np.sqrt(2 * (patch_size**2))
        padded_dist = np.pad(dist_map, pad, mode='constant', constant_values=1.0)

    start_time = time.perf_counter()
    while open_set:
        f, g, _, current = heapq.heappop(open_set)
        if current == goal:
            return nodes_expanded, g, time.perf_counter() - start_time, total_inf_time
            
        if g > g_score.get(current, float('inf')): continue
        nodes_expanded += 1
        
        for neighbor, cost in get_neighbors_8way(grid, current):
            tentative_g = g_score[current] + cost
            if tentative_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = tentative_g
                
                if cnn_model is not None:
                    inf_start = time.perf_counter()
                    px, py = neighbor[0], neighbor[1]
                    patch_obs = padded_grid[px:px+patch_size, py:py+patch_size]
                    patch_dist = padded_dist[px:px+patch_size, py:py+patch_size]
                    tensor_in = torch.tensor(np.stack([patch_obs, patch_dist], axis=0), dtype=torch.float32).unsqueeze(0)
                    with torch.no_grad():
                        h_cnn = cnn_model(tensor_in).item()
                    total_inf_time += (time.perf_counter() - inf_start)
                    h_eucl = euclidean_heuristic(neighbor, goal)
                    h_used = max(h_cnn, h_eucl) 
                else:
                    h_used = euclidean_heuristic(neighbor, goal)
                
                f_score = tentative_g + epsilon * h_used
                heapq.heappush(open_set, (f_score, tentative_g, counter, neighbor))
                counter += 1
    return nodes_expanded, -1, time.perf_counter() - start_time, total_inf_time

# 5. ГЛАВНЫЙ ЦИКЛ ЭКСПЕРИМЕНТА
def run_experiment():
    rng = np.random.default_rng(SEED)
    start, goal = (0,0), (GRID_SIZE-1, GRID_SIZE-1)
    
    results = {d: {'wa_time': [], 'lh_time': [], 'wa_len': [], 'lh_len': []} for d in DENSITIES}
    
    for d in DENSITIES:
        print(f"\n--- Плотность {d*100:.0f}% ---")
        pool_maps = []
        pool_h_stars = []
        
        # Генерация пула карт
        attempts = 0
        while len(pool_maps) < TOTAL_MAPS_PER_DENSITY and attempts < TOTAL_MAPS_PER_DENSITY * 10:
            grid = generate_map(GRID_SIZE, d, rng)
            if is_reachable_8way(grid, start, goal):
                pool_maps.append(grid)
                pool_h_stars.append(compute_dijkstra_heuristic(grid, goal))
            attempts += 1
            
        indices = np.arange(len(pool_maps))
        rng.shuffle(indices)
        split_idx = int(len(pool_maps) * (1 - TEST_FRACTION))
        
        train_maps = [pool_maps[i] for i in indices[:split_idx]]
        train_h_stars = [pool_h_stars[i] for i in indices[:split_idx]]
        test_maps = [pool_maps[i] for i in indices[split_idx:]]
        
        print(f"Сгенерировано {len(pool_maps)} карт. Обучение: {len(train_maps)}, Тест: {len(test_maps)}")
        
        # Обучение CNN ТОЛЬКО на train_maps
        print("Обучение CNN...")
        dataset = HeuristicDataset(train_maps, train_h_stars, PATCH_SIZE)
        dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
        cnn = LightweightCNN().to(DEVICE)
        optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        
        cnn.train()
        for epoch in range(15):
            total_loss = 0
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                optimizer.zero_grad()
                outputs = cnn(inputs).squeeze()
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
        cnn.eval()
        
        # Тестирование ТОЛЬКО на test_maps (Hold-out)
        print("Тестирование на hold-out картах...")
        for grid in tqdm(test_maps, desc=f"Тест {d*100:.0f}%"):
            _, l_wa, t_wa, _ = astar_search(grid, start, goal, epsilon=1.05)
            _, l_lh, t_lh, _ = astar_search(grid, start, goal, epsilon=1.0, cnn_model=cnn, patch_size=PATCH_SIZE)
            
            if l_wa > 0 and l_lh > 0:
                results[d]['wa_time'].append(t_wa * 1000)
                results[d]['lh_time'].append(t_lh * 1000)
                results[d]['wa_len'].append(l_wa)
                results[d]['lh_len'].append(l_lh)
                
    return results

# 6. КОРРЕКТНАЯ СТАТИСТИКА 
def analyze_results(results):
    print("\n СТАТИСТИЧЕСКИЙ АНАЛИЗ ")
    for d in DENSITIES:
        t_lh = np.array(results[d]['lh_time'])
        t_wa = np.array(results[d]['wa_time'])
        l_lh = np.array(results[d]['lh_len'])
        l_wa = np.array(results[d]['wa_len'])
        
        # Отношения (Ratios)
        ratios_t = t_lh / t_wa
        ratios_l = l_lh / l_wa
        
        # Медиана и 95% CI 
        median_t = np.median(ratios_t)
        bootstrap_medians = [np.median(np.random.choice(ratios_t, size=len(ratios_t), replace=True)) for _ in range(1000)]
        ci_t = np.percentile(bootstrap_medians, [2.5, 97.5])
        
        median_l = np.median(ratios_l)
        
        print(f"Плотность {d*100:.0f}% | "
              f"Медиана T_LH/T_WA: {median_t:.3f} (95% CI: [{ci_t[0]:.3f}, {ci_t[1]:.3f}]) | "
              f"Медиана L_LH/L_WA: {median_l:.3f}")
              
        if ci_t[1] < 0.85:
            print("  Гипотеза об ускорении >15% ПОДТВЕРЖДЕНА.")
        else:
            print("  Гипотеза об ускорении >15% НЕ ПОДТВЕРЖДЕНА (накладные расходы инференса превышают выигрыш).")

if __name__ == "__main__":
    exp_results = run_experiment()
    analyze_results(exp_results)

 Реальное число параметров CNN: 9,585

--- Плотность 10% ---
Сгенерировано 2500 карт. Обучение: 2000, Тест: 500
Обучение CNN...


KeyboardInterrupt: 